In [0]:
silver_df = spark.table("silver_customer_products")

schema = silver_df.schema

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("department_id", IntegerType()),
    StructField("aisle_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("add_to_cart_order", IntegerType()),
    StructField("reordered", IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("eval_set", StringType()),
    StructField("order_number", IntegerType()),
    StructField("order_dow", IntegerType()),
    StructField("order_hour_of_day", IntegerType()),
    StructField("days_since_prior_order", DoubleType()),
    StructField("product_name", StringType()),
    StructField("aisle", StringType()),
    StructField("department", StringType())
])

In [0]:
stream_path = "/Volumes/workspace/default/e-commerce_recommendation/incoming_orders"

stream_df = (
    spark.readStream
         .schema(schema)
         .option("header", True)
         .csv(stream_path)
)

In [0]:
from pyspark.sql.functions import count

trending = (
    stream_df
    .groupBy("product_id", "product_name")
    .agg(count("*").alias("purchases"))
)

In [0]:
query = (
    trending.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("trending_products")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()

In [0]:
spark.sql("""
SELECT product_id,
       COUNT(DISTINCT product_name) AS names
FROM trending_products
GROUP BY product_id
HAVING names > 1
""").show(20, False)

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM trending_products
""").show()

In [0]:
spark.sql("""
SELECT *
FROM trending_products
ORDER BY purchases DESC
LIMIT 10
""").show(10,False)

In [0]:
spark.sql("""
SELECT *
FROM trending_products
WHERE product_name='Banana'
""").show(20,False)

In [0]:
spark.sql("SHOW TABLES").show(100, False)

In [0]:
spark.sql("""

SELECT *
FROM trending_products
LIMIT 10

""").show(20,False)

In [0]:
top_products = spark.sql("""

SELECT *
FROM trending_products
ORDER BY purchases DESC
LIMIT 10

""")

display(top_products)

In [0]:
trending_batch = (
    spark.table("trending_products")
)

trending_batch.write.mode("overwrite").saveAsTable(
    "top_trending_products"
)

In [0]:
spark.sql("SHOW TABLES").show()